# Batch Jump-Height Calculation

Loops over every pose-data text file in a folder, runs the jump-height
calculation on each, and collects the results into one table you can save.

**Before running:** start Jupyter from your project root so that
`from utilities.utils import ...`, `import utilities.PTM`, etc. resolve.


## 1. Imports

In [18]:
import glob
import os
import traceback

import numpy as np
import pandas as pd
from scipy.signal import savgol_filter

# Your project modules (run the notebook from the project root)
from utilities.utils import flip_axis, ts_jump_height
import utilities.PTM as PTM

%matplotlib inline

## 2. Configuration

Point `DATA_DIR` at the folder holding your jump text files and set the
glob pattern to match them. `FPS` should match the frame rate the videos
were recorded / sampled at.

In [19]:
DATA_DIR = "keypoints/cmj"      # <-- folder containing your text files
PATTERN  = "*.txt"           # <-- e.g. "*.txt" or "*.csv"
FPS      = 30                # <-- frame rate of the source video

files = sorted(glob.glob(os.path.join(DATA_DIR, PATTERN)))
print(f"Found {len(files)} file(s):")
for f in files:
    print("  ", f)

Found 64 file(s):
   keypoints/cmj\P03_CMJBL_FRONT.txt
   keypoints/cmj\P03_CMJBL_FRONT_world.txt
   keypoints/cmj\P03_CMJUL_FRONT.txt
   keypoints/cmj\P03_CMJUL_FRONT_world.txt
   keypoints/cmj\P04_CMJ_BL_FRONT.txt
   keypoints/cmj\P04_CMJ_BL_FRONT_world.txt
   keypoints/cmj\P04_CMJ_UL_FRONT.txt
   keypoints/cmj\P04_CMJ_UL_FRONT_world.txt
   keypoints/cmj\P05_CMJ_BL_FRONT.txt
   keypoints/cmj\P05_CMJ_BL_FRONT_world.txt
   keypoints/cmj\P05_CMJ_UL_FRONT.txt
   keypoints/cmj\P05_CMJ_UL_FRONT_world.txt
   keypoints/cmj\P06_CMJ_BL_FRONT.txt
   keypoints/cmj\P06_CMJ_BL_FRONT_world.txt
   keypoints/cmj\P06_CMJ_UL_FRONT.txt
   keypoints/cmj\P06_CMJ_UL_FRONT_world.txt
   keypoints/cmj\P07_CMJ_BL_FRONT.txt
   keypoints/cmj\P07_CMJ_BL_FRONT_world.txt
   keypoints/cmj\P07_CMJ_UL_FRONT.txt
   keypoints/cmj\P07_CMJ_UL_FRONT_world.txt
   keypoints/cmj\P08_CMJ_BL_FRONT.txt
   keypoints/cmj\P08_CMJ_BL_FRONT_world.txt
   keypoints/cmj\P08_CMJ_UL_FRONT.txt
   keypoints/cmj\P08_CMJ_UL_FRONT_world.txt
  

## 3. File loader

MediaPipe exports come in a few flavours (comma, tab, or space separated).
`sep=None` with the python engine lets pandas sniff the delimiter, so this
handles most cases. It also checks the columns the calculation needs are
actually present, and tells you which clip is malformed if not.

In [20]:
REQUIRED_COLS = [
    "LEFT_HIP_x", "LEFT_HIP_y",
    "RIGHT_HIP_x", "RIGHT_HIP_y",
    "RIGHT_FOOT_INDEX_x", "RIGHT_FOOT_INDEX_y",
]

def load_pose_file(path):
    """Read a pose-data text file into a DataFrame and validate columns."""
    df = pd.read_csv(path, sep=None, engine="python")
    missing = [c for c in REQUIRED_COLS if c not in df.columns]
    if missing:
        raise ValueError(f"missing columns: {missing} (found: {list(df.columns)})")
    return df

## 4. Jump-height calculation

Same logic as your `calculate_jump`, with three small changes for batch use:

- `fps` is actually passed through to `PTM.mm_per_px` (your version
  hardcoded `30` there).
- `verbose` / `plot` are arguments and default to `False`, so the batch run
  stays quiet. Flip them on when debugging a single clip.


In [21]:
def calculate_jump(data, fps=30, verbose=False, plot=False):
    # Setup hip (midpoint of left/right)
    hip_x = (data["LEFT_HIP_x"] + data["RIGHT_HIP_x"]) / 2
    hip_y = (data["LEFT_HIP_y"] + data["RIGHT_HIP_y"]) / 2
    hip = np.column_stack((hip_x, hip_y))

    # Setup toe
    toe = data[["RIGHT_FOOT_INDEX_x", "RIGHT_FOOT_INDEX_y"]].to_numpy()

    # Flip axis
    hip_flipped = flip_axis(hip.copy())
    toe_flipped = flip_axis(toe.copy())

    # Smooth before mm_per_px to avoid noise peaks
    ts = savgol_filter(hip_flipped[:, 1], window_length=11, polyorder=2)

    # Calculate pixel-to-mm scale using gravity
    mm_px = PTM.mm_per_px(ts, fps=fps, verbose=verbose, plot=plot)

    toe_y_mm = toe_flipped[:, 1] * mm_px
    return ts_jump_height(toe_y_mm, fps=fps)

## 5. Run over every file

Each file is wrapped in try/except so one bad clip won't stop the whole
batch — failures are recorded with their error message instead.

In [22]:
results = []

for path in files:
    if path.endswith(".txt") and "world" not in path:
        name = os.path.basename(path)
        try:
            data = load_pose_file(path)
            jh = calculate_jump(data, fps=FPS, verbose=False, plot=False)
            results.append({"file": name, "jump_height": jh, "error": ""})
            print(f"OK   {name:40s}  jump_height = {jh}")
        except Exception as e:
            results.append({"file": name, "jump_height": np.nan, "error": str(e)})
            print(f"FAIL {name:40s}  {e}")
            # Uncomment for full traceback while debugging:
            # traceback.print_exc()

print(f"\nProcessed {len(results)} file(s).")

FAIL P03_CMJBL_FRONT.txt                       missing columns: ['LEFT_HIP_x', 'LEFT_HIP_y', 'RIGHT_HIP_x', 'RIGHT_HIP_y', 'RIGHT_FOOT_INDEX_x', 'RIGHT_FOOT_INDEX_y'] (found: ['0.5353298783302307', '0.43739670515060425', '-0.36792150139808655'])
FAIL P03_CMJUL_FRONT.txt                       missing columns: ['LEFT_HIP_x', 'LEFT_HIP_y', 'RIGHT_HIP_x', 'RIGHT_HIP_y', 'RIGHT_FOOT_INDEX_x', 'RIGHT_FOOT_INDEX_y'] (found: ['0.5241571068763733', '0.43820852041244507', '-0.218971386551857'])
FAIL P04_CMJ_BL_FRONT.txt                      missing columns: ['LEFT_HIP_x', 'LEFT_HIP_y', 'RIGHT_HIP_x', 'RIGHT_HIP_y', 'RIGHT_FOOT_INDEX_x', 'RIGHT_FOOT_INDEX_y'] (found: ['0.507747232913971', '0.4126063287258148', '-0.32495173811912537'])
FAIL P04_CMJ_UL_FRONT.txt                      missing columns: ['LEFT_HIP_x', 'LEFT_HIP_y', 'RIGHT_HIP_x', 'RIGHT_HIP_y', 'RIGHT_FOOT_INDEX_x', 'RIGHT_FOOT_INDEX_y'] (found: ['0.5167849659919739', '0.4166264533996582', '-0.3291928470134735'])
FAIL P05_CMJ_BL_FRONT.

## 6. Results table

In [23]:
df_results = pd.DataFrame(results)
df_results
df_results.describe()

,jump_height
count,0.0
mean,NaN
std,NaN
min,NaN
25%,NaN
50%,NaN
75%,NaN
max,NaN


## 7. Save results

Writes a CSV next to your data. Adjust the path if you want it elsewhere.

In [24]:
out_path = "jump_heights.csv"
df_results.to_csv(out_path, index=False)
print(f"Saved -> {out_path}")

Saved -> jump_heights.csv


## 8. Debug a single clip (optional)

When a file fails or a number looks off, run it on its own with the plot and
verbose output turned back on to see the gravity fit.

In [25]:
debug_file = files[0] if files else None  # or set a specific path

if debug_file:
    data = load_pose_file(debug_file)
    jh = calculate_jump(data, fps=FPS, verbose=True, plot=True)
    print(f"{os.path.basename(debug_file)}: jump_height = {jh}")

ValueError: missing columns: ['LEFT_HIP_x', 'LEFT_HIP_y', 'RIGHT_HIP_x', 'RIGHT_HIP_y', 'RIGHT_FOOT_INDEX_x', 'RIGHT_FOOT_INDEX_y'] (found: ['0.5353298783302307', '0.43739670515060425', '-0.36792150139808655'])